In [ ]:
# ==========================================
# WEIGHT TUNING FOR GATE TF PREDICTOR
# ==========================================

import pandas as pd

from itertools import product

from scipy import stats
from sklearn.preprocessing import MinMaxScaler

# ==========================================
# LOAD DATA
# ==========================================

csv_path = r"D:\GATE_topic_predictor\topic_frequency.csv"

df = pd.read_csv(csv_path)

# ==========================================
# UNIQUE YEARS
# ==========================================

years = sorted(

    df["Year"].astype(str).unique()

)

# ==========================================
# WEIGHT SEARCH SPACE
# ==========================================

weights = [

    0.1,

    0.2,

    0.3,

    0.4,

    0.5,

    0.6,

    0.7

]

# ==========================================
# STORE BEST RESULTS
# ==========================================

best_accuracy = 0

best_weights = None

all_results = []

# ==========================================
# GRID SEARCH
# ==========================================

for w1, w2, w3 in product(

    weights,

    repeat=3

):

    # --------------------------------------
    # WEIGHTS MUST SUM TO 1
    # --------------------------------------

    if round(w1 + w2 + w3, 1) == 1.0:

        validation_accuracies = []

        # ==================================
        # ROLLING VALIDATION
        # ==================================

        for i in range(5, len(years) - 1):

            # ----------------------------------
            # TRAIN YEARS
            # ----------------------------------

            train_years = years[:i]

            # ----------------------------------
            # TEST YEAR
            # ----------------------------------

            test_year = years[i]

            # ----------------------------------
            # TRAIN DATA
            # ----------------------------------

            train_df = df[

                df["Year"].astype(str).isin(
                    train_years
                )

            ]

            # ----------------------------------
            # TEST DATA
            # ----------------------------------

            test_df = df[

                df["Year"].astype(str) == test_year

            ]

            # ----------------------------------
            # TOTAL FREQUENCY
            # ----------------------------------

            total_frequency = train_df.groupby(
                "Topic"
            )["Frequency"].sum()

            # ----------------------------------
            # RECENT YEARS
            # ----------------------------------

            recent_years = train_years[-3:]

            recent_df = train_df[

                train_df["Year"].astype(str).isin(
                    recent_years
                )

            ]

            recent_frequency = recent_df.groupby(
                "Topic"
            )["Frequency"].sum()

            # ----------------------------------
            # TREND SCORES
            # ----------------------------------

            pivot_df = train_df.pivot_table(

                index="Year",

                columns="Topic",

                values="Frequency",

                aggfunc="sum"

            ).fillna(0)

            trend_scores = {}

            for topic in pivot_df.columns:
                slope, _, _, _, _ = stats.linregress(
                    range(len(pivot_df[topic])),
                    pivot_df[topic].values
                )
                trend_scores[topic] = slope

        

            # ----------------------------------
            # CREATE FEATURE TABLE
            # ----------------------------------

            prediction_results = []

            for topic in total_frequency.index:

                freq_score = total_frequency[
                    topic
                ]

                trend_score = trend_scores.get(
                    topic,
                    0
                )

                recent_score = recent_frequency.get(
                    topic,
                    0
                )

                prediction_results.append({

                    "Topic": topic,

                    "Frequency": freq_score,

                    "Trend": trend_score,

                    "Recent": recent_score

                })

            # ----------------------------------
            # DATAFRAME
            # ----------------------------------

            prediction_df = pd.DataFrame(
                prediction_results
            )

            # ----------------------------------
            # NORMALIZATION
            # ----------------------------------

            scaler = MinMaxScaler()

            cols = [

                "Frequency",

                "Trend",

                "Recent"

            ]

            prediction_df[cols] = scaler.fit_transform(

                prediction_df[cols]

            )

            # ----------------------------------
            # PREDICTION SCORE
            # ----------------------------------

            prediction_df["Prediction Score"] = (

                prediction_df["Frequency"] * w1 +

                prediction_df["Trend"] * w2 +

                prediction_df["Recent"] * w3

            )

            # ----------------------------------
            # SORT TOPICS
            # ----------------------------------

            prediction_df = prediction_df.sort_values(

                by="Prediction Score",

                ascending=False

            )

            # ----------------------------------
            # TOP K
            # ----------------------------------

            TOP_K = 5

            predicted_topics = set(

                prediction_df.head(TOP_K)["Topic"]

            )

            # ----------------------------------
            # ACTUAL TOPICS
            # ----------------------------------

            actual_topic_df = test_df.groupby(
                "Topic"
            )["Frequency"].sum().reset_index()

            actual_topic_df = actual_topic_df.sort_values(

                by="Frequency",

                ascending=False

            )

            actual_topics = set(

                actual_topic_df.head(TOP_K)["Topic"]

            )

            # ----------------------------------
            # ACCURACY
            # ----------------------------------

            correct_predictions = predicted_topics.intersection(

                actual_topics

            )

            accuracy = (

                len(correct_predictions)

                / TOP_K

            ) * 100

            validation_accuracies.append(
                accuracy
            )

        # ==================================
        # AVERAGE ACCURACY
        # ==================================

        average_accuracy = sum(

            validation_accuracies

        ) / len(validation_accuracies)

        # ==================================
        # STORE RESULTS
        # ==================================

        all_results.append({

            "Frequency Weight": w1,

            "Trend Weight": w2,

            "Recent Weight": w3,

            "Accuracy": round(
                average_accuracy,
                2
            )

        })

        # ==================================
        # BEST WEIGHTS
        # ==================================

        if average_accuracy > best_accuracy:

            best_accuracy = average_accuracy

            best_weights = (

                w1,

                w2,

                w3

            )

# ==========================================
# RESULTS DATAFRAME
# ==========================================

results_df = pd.DataFrame(
    all_results
)

# ==========================================
# SORT RESULTS
# ==========================================

results_df = results_df.sort_values(

    by="Accuracy",

    ascending=False

)

# ==========================================
# DISPLAY BEST RESULT
# ==========================================

print("\n===================================")
print("BEST WEIGHT COMBINATION")
print("===================================")

print(f"\nFrequency Weight: {best_weights[0]}")

print(f"Trend Weight: {best_weights[1]}")

print(f"Recent Weight: {best_weights[2]}")

print(f"\nBest Accuracy: {best_accuracy:.2f}%")

# ==========================================
# TOP RESULTS
# ==========================================

print("\n===================================")
print("TOP 10 WEIGHT COMBINATIONS")
print("===================================")

print(results_df.head(10))

# ==========================================
# SAVE CSV
# ==========================================

output_path = r"D:\TF_Gate_Topic_predictor\data\weight_tuning_results.csv"

results_df.to_csv(
    output_path,
    index=False
)

print("\nResults saved successfully!")
print(output_path)



BEST WEIGHT COMBINATION

Frequency Weight: 0.6
Trend Weight: 0.1
Recent Weight: 0.3

Best Accuracy: 80.00%

TOP 10 WEIGHT COMBINATIONS
    Frequency Weight  Trend Weight  Recent Weight  Accuracy
28               0.6           0.1            0.3      80.0
31               0.7           0.1            0.2      80.0
32               0.7           0.2            0.1      80.0
29               0.6           0.2            0.2      80.0
25               0.5           0.2            0.3      78.0
24               0.5           0.1            0.4      78.0
30               0.6           0.3            0.1      78.0
20               0.4           0.2            0.4      74.0
6                0.2           0.1            0.7      74.0
19               0.4           0.1            0.5      74.0

Results saved successfully!
D:\GATE_topic_predictor\weight_tuning_results.csv
